# Proyecto 02
## Realización de modelos - Sentence-BERT

**Carnet/Autores:**
- 22473, Madeline Nahomy Castro Morales
- 22716, Aroldo Xavier López Osoy
- 22309, Diego Pablo Valenzuela Palacios
- 22281, Gerson Alexander Ramirez Conoz

**Catedrático:** Mario Barrientos

**Sección:** 30

**Fecha:** 23/10/2025

---

### Importaciones

In [27]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import warnings
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')

### Carga de Datos Preprocesados

In [28]:
DATA_DIR = Path("../data")
PP_DIR = DATA_DIR / "preprocessed"
TRAIN_PP = PP_DIR / "train_preprocessed.csv"
TEST_PP = PP_DIR / "test_preprocessed.csv"
LABEL_MAPS = PP_DIR / "label_maps.json"

OUTPUT_DIR = Path("./model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(TRAIN_PP)
test = pd.read_csv(TEST_PP)

with open(LABEL_MAPS, "r", encoding="utf-8") as f:
    label_maps = json.load(f)

print(f"Datos cargados - Train: {train.shape}, Test: {test.shape}")

Datos cargados - Train: (36696, 15), Test: (3, 11)


### Preparación de Datos para Modelo Jerárquico

In [29]:
# Texto combinado para embeddings
def create_input_text(row):
    """Combina Question + Answer + Explanation para sBERT"""
    parts = []
    if pd.notna(row.get('QuestionText')):
        parts.append(str(row['QuestionText']))
    if pd.notna(row.get('MC_Answer')):
        parts.append(f"Answer: {row['MC_Answer']}")
    if pd.notna(row.get('StudentExplanation_clean')):
        parts.append(f"Explanation: {row['StudentExplanation_clean']}")
    return " ".join(parts)

train['input_text'] = train.apply(create_input_text, axis=1)
test['input_text'] = test.apply(create_input_text, axis=1)

print(f"Texto de entrada creado")
print(f"Ejemplo: {train['input_text'].iloc[0][:150]}...")


Texto de entrada creado
Ejemplo: What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them ar...


In [30]:
# Separar datos según Category
# Para Category con Misconception
train_misconception = train[train['Category'].str.contains('Misconception', na=False)].copy()
train_misconception = train_misconception[train_misconception['Misconception'].notna()]

print(f"\nDatos con Misconception: {len(train_misconception)} registros")
print(f"Categorías totales: {train['Category'].nunique()}")
print(f"Misconceptions totales: {train_misconception['Misconception'].nunique()}")


Datos con Misconception: 9860 registros
Categorías totales: 6
Misconceptions totales: 35


### Generación de Embeddings con sBERT

In [31]:
# modelo sBERT pre-entrenado
model_name = 'all-MiniLM-L6-v2'  
print(f"Cargando modelo: {model_name}")
sbert_model = SentenceTransformer(model_name)
print(f"Modelo sBERT cargado - Dimensión embeddings: {sbert_model.get_sentence_embedding_dimension()}")

# Generación de embeddings para train
print("\nGenerando embeddings para train...")
X_train_embeddings = sbert_model.encode(
    train['input_text'].tolist(),
    show_progress_bar=True,
    batch_size=16,
    convert_to_numpy=True
)
print(f"Embeddings train: {X_train_embeddings.shape}")

# Generar embeddings para train_misconception
if len(train_misconception) > 0:
    print("\nGenerando embeddings para misconceptions...")
    X_train_misc_embeddings = sbert_model.encode(
        train_misconception['input_text'].tolist(),
        show_progress_bar=True,
        batch_size=16,
        convert_to_numpy=True
    )
    print(f"Embeddings misconception: {X_train_misc_embeddings.shape}")

# Generar embeddings para test
print("\nGenerando embeddings para test...")
X_test_embeddings = sbert_model.encode(
    test['input_text'].tolist(),
    show_progress_bar=True,
    batch_size=16,
    convert_to_numpy=True
)
print(f"Embeddings test: {X_test_embeddings.shape}")

Cargando modelo: all-MiniLM-L6-v2
Modelo sBERT cargado - Dimensión embeddings: 384

Generando embeddings para train...


Batches: 100%|██████████| 2294/2294 [07:00<00:00,  5.46it/s]


Embeddings train: (36696, 384)

Generando embeddings para misconceptions...


Batches: 100%|██████████| 617/617 [01:49<00:00,  5.61it/s]


Embeddings misconception: (9860, 384)

Generando embeddings para test...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.95it/s]

Embeddings test: (3, 384)


### Preparación de labels 

In [36]:
# Encoder para Category
label_encoder_category = LabelEncoder()
y_train_category = label_encoder_category.fit_transform(train['Category'])
print(f"Categories encoded: {len(label_encoder_category.classes_)} clases")

# Encoder para Misconception
if len(train_misconception) > 0:
    label_encoder_misconception = LabelEncoder()
    y_train_misconception = label_encoder_misconception.fit_transform(
        train_misconception['Misconception']
    )
    print(f"Misconceptions encoded: {len(label_encoder_misconception.classes_)} clases")


Categories encoded: 6 clases
Misconceptions encoded: 35 clases


### División de Train/Test 

In [37]:
X_train_final, X_val, y_train_cat_final, y_val_cat = train_test_split(
    X_train_embeddings, 
    y_train_category,
    test_size=0.2,  
    random_state=42,
    stratify=y_train_category  
)

print(f"Train final: {X_train_final.shape[0]} muestras")
print(f"Validation: {X_val.shape[0]} muestras")

Train final: 29356 muestras
Validation: 7340 muestras


In [38]:
if len(train_misconception) > 0:
    print("\nDividiendo datos de Misconception...")
    
    X_train_misc_final, X_val_misc, y_train_misc_final, y_val_misc = train_test_split(
        X_train_misc_embeddings,
        y_train_misconception,
        test_size=0.2,
        random_state=42,
        stratify=y_train_misconception
    )
    
    print(f"Train Misconception: {X_train_misc_final.shape[0]} muestras")
    print(f"Val Misconception: {X_val_misc.shape[0]} muestras")


Dividiendo datos de Misconception...
Train Misconception: 7888 muestras
Val Misconception: 1972 muestras


### Construcción y evaluación de modelos

In [39]:
print("\nEntrenando SVM Category...")
base_svm = LinearSVC(C=1.0, random_state=42, class_weight='balanced', max_iter=1000, dual='auto')
category_model_svm = CalibratedClassifierCV(base_svm, cv=3, n_jobs=1)
category_model_svm.fit(X_train_final, y_train_cat_final) 
y_pred_svm = category_model_svm.predict(X_val)
acc_svm = accuracy_score(y_val_cat, y_pred_svm)
print(f"SVM - Accuracy: {acc_svm:.4f} ({acc_svm*100:.2f}%)")


Entrenando SVM Category...
SVM - Accuracy: 0.7151 (71.51%)


In [40]:
print("\nEntrenando Random Forest...")
category_model_rf = RandomForestClassifier(
    n_estimators=100, max_depth=20, min_samples_split=5,
    random_state=42, class_weight='balanced', n_jobs=-1
)
category_model_rf.fit(X_train_final, y_train_cat_final)  
y_pred_rf = category_model_rf.predict(X_val)
acc_rf = accuracy_score(y_val_cat, y_pred_rf)
print(f"Random Forest - Accuracy: {acc_rf:.4f} ({acc_rf*100:.2f}%)")


Entrenando Random Forest...
Random Forest - Accuracy: 0.8165 (81.65%)


In [41]:
print("\nEntrenando MLP...")
category_model_mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
    alpha=0.0001, batch_size=128, learning_rate='adaptive', max_iter=30,
    random_state=42, early_stopping=True, validation_fraction=0.1, n_iter_no_change=5
)
category_model_mlp.fit(X_train_final, y_train_cat_final)  
y_pred_mlp = category_model_mlp.predict(X_val)
acc_mlp = accuracy_score(y_val_cat, y_pred_mlp)
print(f"MLP - Accuracy: {acc_mlp:.4f} ({acc_mlp*100:.2f}%)")


Entrenando MLP...
MLP - Accuracy: 0.8178 (81.78%)


In [42]:
misconception_model_final = None
if len(train_misconception) > 0:
    print("\nEntrenando Misconception...")
    base_misc_svm = LinearSVC(C=1.0, random_state=42, class_weight='balanced', max_iter=1000, dual='auto')
    misconception_model_final = CalibratedClassifierCV(base_misc_svm, cv=3, n_jobs=1)
    misconception_model_final.fit(X_train_misc_final, y_train_misc_final)  
    y_pred_misc = misconception_model_final.predict(X_val_misc)
    acc_misc = accuracy_score(y_val_misc, y_pred_misc)
    print(f"Misconception - Accuracy: {acc_misc:.4f} ({acc_misc*100:.2f}%)")


Entrenando Misconception...
Misconception - Accuracy: 0.9447 (94.47%)


### Función de predicción jerárquica

In [43]:
def hierarchical_predict_top_k(features, category_model, misconception_model, 
                                 category_encoder, misconception_encoder, top_k=3):
    """
    Pipeline jerárquico para generar predicciones Category:Misconception.
    
    Pasos:
    1. Predecir Category con probabilidades
    2. Para cada Category predicha:
       - Si es *_Misconception: predecir Misconception específica
       - Si no es *_Misconception: usar 'NA'
    3. Generar top-k combinaciones ordenadas por probabilidad
    """
    n_samples = features.shape[0]
    predictions = []
    
    # Predecir Category con probabilidades
    category_probs = category_model.predict_proba(features)
    
    for i in range(n_samples):
        sample_predictions = []
        
        # Obtener top categorías ordenadas por probabilidad
        top_category_indices = np.argsort(category_probs[i])[::-1]
        
        for cat_idx in top_category_indices:
            category = category_encoder.inverse_transform([cat_idx])[0]
            cat_prob = category_probs[i][cat_idx]
            
            # Verificar si esta categoría requiere Misconception
            if 'Misconception' in category:
                if misconception_model is not None:
                    # Obtener probabilidades de misconceptions
                    misc_probs = misconception_model.predict_proba(features[i:i+1])[0]
                    
                    # Top 3 misconceptions
                    top_misc_indices = np.argsort(misc_probs)[::-1][:3]
                    
                    for misc_idx in top_misc_indices:
                        misconception = misconception_encoder.inverse_transform([misc_idx])[0]
                        combined_prob = cat_prob * misc_probs[misc_idx]
                        
                        sample_predictions.append({
                            'label': f"{category}:{misconception}",
                            'prob': combined_prob
                        })
                else:
                    sample_predictions.append({
                        'label': f"{category}:NA",
                        'prob': cat_prob
                    })
            else:
                sample_predictions.append({
                    'label': f"{category}:NA",
                    'prob': cat_prob
                })
        
        # Ordenar por probabilidad y tomar top-k
        sample_predictions = sorted(sample_predictions, key=lambda x: x['prob'], reverse=True)
        top_k_labels = [pred['label'] for pred in sample_predictions[:top_k]]
        
        predictions.append(top_k_labels)
    
    return predictions

### Generación de predicciones para los 3 modelos

In [44]:
# Modelo 1: SVM
print("\nPredicciones con SVM...")
test_predictions_svm = hierarchical_predict_top_k(
    X_test_embeddings,
    category_model_svm,
    misconception_model_final,
    label_encoder_category,
    label_encoder_misconception if len(train_misconception) > 0 else None,
    top_k=3
)
print(f"Predicciones SVM generadas: {len(test_predictions_svm)} muestras")

# Modelo 2: Random Forest
print("\nPredicciones con Random Forest...")
test_predictions_rf = hierarchical_predict_top_k(
    X_test_embeddings,
    category_model_rf,
    misconception_model_final,
    label_encoder_category,
    label_encoder_misconception if len(train_misconception) > 0 else None,
    top_k=3
)
print(f"Predicciones RF generadas: {len(test_predictions_rf)} muestras")

# Modelo 3: MLP
print("\nPredicciones con MLP...")
test_predictions_mlp = hierarchical_predict_top_k(
    X_test_embeddings,
    category_model_mlp,
    misconception_model_final,
    label_encoder_category,
    label_encoder_misconception if len(train_misconception) > 0 else None,
    top_k=3
)
print(f"Predicciones MLP generadas: {len(test_predictions_mlp)} muestras")



Predicciones con SVM...
Predicciones SVM generadas: 3 muestras

Predicciones con Random Forest...
Predicciones RF generadas: 3 muestras

Predicciones con MLP...
Predicciones MLP generadas: 3 muestras


### Ejemplos de preddiciones por cada modelo

In [45]:
print(f"  SVM  - Top-3: {test_predictions_svm[0]}")
print(f"  RF   - Top-3: {test_predictions_rf[0]}")
print(f"  MLP  - Top-3: {test_predictions_mlp[0]}")

if len(test_predictions_svm) > 1:
    print(f"  SVM  - Top-3: {test_predictions_svm[1]}")
    print(f"  RF   - Top-3: {test_predictions_rf[1]}")
    print(f"  MLP  - Top-3: {test_predictions_mlp[1]}")

  SVM  - Top-3: ['True_Correct:NA', 'True_Neither:NA', 'False_Neither:NA']
  RF   - Top-3: ['True_Correct:NA', 'True_Neither:NA', 'False_Neither:NA']
  MLP  - Top-3: ['True_Correct:NA', 'True_Neither:NA', 'False_Correct:NA']
  SVM  - Top-3: ['False_Misconception:WNB', 'False_Neither:NA', 'False_Misconception:Incomplete']
  RF   - Top-3: ['False_Misconception:WNB', 'False_Neither:NA', 'False_Misconception:Incomplete']
  MLP  - Top-3: ['False_Misconception:WNB', 'False_Misconception:Incomplete', 'False_Neither:NA']


### Guardando modelos

In [48]:
with open(OUTPUT_DIR / "category_model_mlp.pkl", "wb") as f:
    pickle.dump(category_model_mlp, f)

with open(OUTPUT_DIR / "misconception_model.pkl", "wb") as f:
    pickle.dump(misconception_model_final, f)

with open(OUTPUT_DIR / "label_encoder_category.pkl", "wb") as f:
    pickle.dump(label_encoder_category, f)

with open(OUTPUT_DIR / "label_encoder_misconception.pkl", "wb") as f:
    pickle.dump(label_encoder_misconception, f)

print("Modelos guardados en:", OUTPUT_DIR)

Modelos guardados en: model_outputs
